## Imports

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Conv2D, Conv2DTranspose, MaxPooling2D, BatchNormalization, Activation, Dropout, concatenate
from keras.optimizers import Adam

# Modified U-Net Architecture

In [ ]:
def unet_64to256(input_shape=(64, 64, 3), n_classes=3, final_activation='sigmoid', dropout_rate=0.05):
    inputs = Input(shape=input_shape)

    # --- Encoder ---
    c1 = Conv2D(16, (3,3), padding='same')(inputs); c1 = BatchNormalization()(c1); c1 = Activation('relu')(c1)
    c1 = Conv2D(16, (3,3), padding='same')(c1); c1 = BatchNormalization()(c1); c1 = Activation('relu')(c1)
    p1 = MaxPooling2D((2,2))(c1); p1 = Dropout(dropout_rate)(p1)  # 64 -> 32

    c2 = Conv2D(32, (3,3), padding='same')(p1); c2 = BatchNormalization()(c2); c2 = Activation('relu')(c2)
    c2 = Conv2D(32, (3,3), padding='same')(c2); c2 = BatchNormalization()(c2); c2 = Activation('relu')(c2)
    p2 = MaxPooling2D((2,2))(c2); p2 = Dropout(dropout_rate)(p2)  # 32 -> 16

    c3 = Conv2D(64, (3,3), padding='same')(p2); c3 = BatchNormalization()(c3); c3 = Activation('relu')(c3)
    c3 = Conv2D(64, (3,3), padding='same')(c3); c3 = BatchNormalization()(c3); c3 = Activation('relu')(c3)
    p3 = MaxPooling2D((2,2))(c3); p3 = Dropout(dropout_rate)(p3)  # 16 -> 8

    c4 = Conv2D(128, (3,3), padding='same')(p3); c4 = BatchNormalization()(c4); c4 = Activation('relu')(c4)
    c4 = Conv2D(128, (3,3), padding='same')(c4); c4 = BatchNormalization()(c4); c4 = Activation('relu')(c4)
    p4 = MaxPooling2D((2,2))(c4); p4 = Dropout(dropout_rate)(p4)  # 8 -> 4

    # --- Bottleneck ---
    c5 = Conv2D(256, (3,3), padding='same')(p4); c5 = BatchNormalization()(c5); c5 = Activation('relu')(c5)
    c5 = Conv2D(256, (3,3), padding='same')(c5); c5 = BatchNormalization()(c5); c5 = Activation('relu')(c5)

    # --- Decoder ---
    u6 = Conv2DTranspose(128, (3,3), strides=(2,2), padding='same')(c5); u6 = concatenate([u6, c4]); u6 = Dropout(dropout_rate)(u6)
    u6 = Conv2D(128, (3,3), padding='same')(u6); u6 = BatchNormalization()(u6); u6 = Activation('relu')(u6)
    u6 = Conv2D(128, (3,3), padding='same')(u6); u6 = BatchNormalization()(u6); u6 = Activation('relu')(u6)

    u7 = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same')(u6); u7 = concatenate([u7, c3]); u7 = Dropout(dropout_rate)(u7)
    u7 = Conv2D(64, (3,3), padding='same')(u7); u7 = BatchNormalization()(u7); u7 = Activation('relu')(u7)
    u7 = Conv2D(64, (3,3), padding='same')(u7); u7 = BatchNormalization()(u7); u7 = Activation('relu')(u7)

    u8 = Conv2DTranspose(32, (3,3), strides=(2,2), padding='same')(u7); u8 = concatenate([u8, c2]); u8 = Dropout(dropout_rate)(u8)
    u8 = Conv2D(32, (3,3), padding='same')(u8); u8 = BatchNormalization()(u8); u8 = Activation('relu')(u8)
    u8 = Conv2D(32, (3,3), padding='same')(u8); u8 = BatchNormalization()(u8); u8 = Activation('relu')(u8)

    u9 = Conv2DTranspose(16, (3,3), strides=(2,2), padding='same')(u8); u9 = concatenate([u9, c1]); u9 = Dropout(dropout_rate)(u9)
    u9 = Conv2D(16, (3,3), padding='same')(u9); u9 = BatchNormalization()(u9); u9 = Activation('relu')(u9)
    u9 = Conv2D(16, (3,3), padding='same')(u9); u9 = BatchNormalization()(u9); u9 = Activation('relu')(u9)

    # Upsample 64->128->256
    u10 = Conv2DTranspose(16, (3,3), strides=(2,2), padding='same')(u9); u10 = Dropout(dropout_rate)(u10)
    u10 = Conv2D(16, (3,3), padding='same')(u10); u10 = BatchNormalization()(u10); u10 = Activation('relu')(u10)

    u11 = Conv2DTranspose(16, (3,3), strides=(2,2), padding='same')(u10); u11 = Dropout(dropout_rate)(u11)
    u11 = Conv2D(16, (3,3), padding='same')(u11); u11 = BatchNormalization()(u11); u11 = Activation('relu')(u11)

    outputs = Conv2D(n_classes, (1,1), activation=final_activation)(u11)

    model = Model(inputs=inputs, outputs=outputs, name='UNet_64to256')
    model.compile(optimizer=Adam(1e-4), loss='mae')
    return model

## Initialize Model

In [ ]:
model = unet_64to256()
model.summary()

## Data Generator

In [ ]:
from google.colab import files
files.upload()  # select kaggle.json

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d jessicali9530/celeba-dataset
!unzip -q celeba-dataset.zip -d ./celeba_images

In [ ]:
import os
import cv2
import numpy as np

# Correct path to images
image_dir = './celeba_images/img_align_celeba/img_align_celeba/'
imgs = [f for f in os.listdir(image_dir) if f.lower().endswith('.jpg')]

print("Number of images found:", len(imgs))
print("First 5 images:", imgs[:5])

def datagen(batch_size):
    while True:
        x_batch = []
        y_batch = []

        for _ in range(batch_size):
            indx = np.random.randint(0, len(imgs))
            path = os.path.join(image_dir, imgs[indx])

            # Read image safely
            bgr = cv2.imread(path)
            if bgr is None:
                continue  # skip if image not read properly

            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

            # High-res 256x256
            y = cv2.resize(rgb, (256, 256)) / 255.0

            # Low-res 64x64
            x = cv2.resize(rgb, (64, 64)) / 255.0

            # Ensure float32
            x = x.astype('float32')
            y = y.astype('float32')

            x_batch.append(x)
            y_batch.append(y)

        yield np.array(x_batch), np.array(y_batch)

### Set Training Parameters

In [ ]:
batch_size = 16
steps_per_epoch = len(imgs) // batch_size
epochs = 3

In [ ]:
def show_and_save_samples(model, imgs, base_dir, epoch, n=5, save_dir='./epoch_samples'):
    os.makedirs(save_dir, exist_ok=True)
    chosen = np.random.choice(imgs, n, replace=False)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = np.expand_dims(axes, 0)

    for i, fname in enumerate(chosen):
        path = os.path.join(base_dir, fname)
        bgr = cv2.imread(path)
        if bgr is None:
            print(f"Skipping {fname}, could not read")
            continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        gt256 = cv2.resize(rgb, (256,256)).astype('float32')/255
        img64 = cv2.resize(rgb, (64,64)).astype('float32')/255
        pred  = model.predict(np.expand_dims(img64,0), verbose=0)[0]

        lowup = cv2.resize((img64*255).astype(np.uint8),(256,256))

        for ax, im, title in zip(axes[i],[rgb,gt256,lowup,pred],
                                 ['Original','GT 256x256','Low-Res Upsampled','Prediction']):
            ax.imshow(im); ax.set_title(title); ax.axis('off')

    plt.tight_layout()
    plt.savefig(f"{save_dir}/epoch_{epoch+1:02d}.png")
    plt.show()

In [ ]:
from tensorflow.keras.callbacks import LambdaCallback

base_dir = './celeba_images/img_align_celeba/img_align_celeba/'

show_cb = LambdaCallback(
    on_epoch_end=lambda epoch, logs: show_and_save_samples(model, imgs, base_dir, epoch, n=3)
)

### Train the model

In [ ]:
history = model.fit(
    datagen(batch_size=batch_size),
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    callbacks=[show_cb],
    verbose=1
)

In [ ]:
# Pick one image for testing
img_path = './celeba_images/img_align_celeba/img_align_celeba/' + imgs[0]
bgr = cv2.imread(img_path)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

img_lr = cv2.resize(rgb, (64,64)) / 255.0
img_lr_input = np.expand_dims(img_lr, 0)

pred_hr = model.predict(img_lr_input)[0]
pred_hr = np.clip(pred_hr, 0, 1)

plt.subplot(1,2,1)
plt.imshow(img_lr); plt.title('Low-Res 64x64'); plt.axis('off')
plt.subplot(1,2,2)
plt.imshow(pred_hr); plt.title('High-Res 256x256'); plt.axis('off')
plt.show()
